# ScanMatrix: Barcode Detection and Decoding Pipeline

## Project Overview

This notebook implements a complete barcode detection and decoding system designed for warehouse and logistics applications. The pipeline combines deep learning based detection with multiple decoding strategies to achieve reliable barcode reading across varying distances and image quality conditions.

---

## Table of Contents

1. Introduction
2. System Architecture
3. Setup and Dependencies
4. Configuration
5. Detection Module
6. Decoding Pipeline
7. Main Processing Function
8. Execution
9. Batch Processing
10. Results and Evaluation

---

## 1. Introduction

### Problem Statement

Barcode scanning in warehouse environments presents several challenges:

* Variable distances between camera and barcode
* Inconsistent lighting conditions
* Motion blur from handheld devices
* Different barcode types (1D linear, 2D matrix codes)
* Barcodes at various angles and orientations

### Our Approach

This pipeline addresses these challenges through a two stage approach:

1. **Detection Stage**: Uses YOLOv8 to locate barcodes within images regardless of their position or size.

2. **Decoding Stage**: Applies multiple preprocessing strategies and decoding algorithms to extract barcode data from detected regions.

### Key Features

* Custom trained YOLO model for barcode detection
* Multi decoder fusion (pyzbar + zxing)
* Adaptive preprocessing strategies
* Support for 1D and 2D barcode formats
* Multi scale detection for distant barcodes
* Rotation handling for angled barcodes
* Batch processing capability

## 2. System Architecture

The pipeline follows this flow:

```
Input Image --> YOLO Detection --> Crop Extraction --> Preprocessing --> Multi Decoder --> Output
```

### Preprocessing Strategies Applied

* **Raw**: No modification, works best for high quality images
* **Upscaling (2x, 3x, 4x)**: Bicubic interpolation for small or distant barcodes
* **CLAHE**: Contrast enhancement for unevenly lit images
* **Sharpening**: Edge enhancement for slightly blurry images
* **Otsu Thresholding**: Automatic binarization to black and white
* **Combined sharpen + 5x upscale**: For difficult cases

### Decoders Used

* **pyzbar**: Fast and reliable for most 1D barcodes and QR codes
* **zxing**: Broader format support, often succeeds where pyzbar fails

Both decoders attempt 90, 180, and 270 degree rotations for angled barcodes.

## 3. Setup and Dependencies

### Required Packages

| Package | Purpose |
|---------|--------|
| ultralytics | YOLOv8 object detection framework |
| opencv-python | Image processing |
| pyzbar | Barcode decoding (ZBar wrapper) |
| zxing-cpp | Alternative decoder with broader format support |
| numpy | Numerical operations |
| matplotlib | Visualization |
| torch | GPU acceleration |

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Install packages
%pip install ultralytics opencv-python pyzbar zxing-cpp matplotlib numpy torch --quiet
!apt-get install -y libzbar0 --quiet

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
from time import time
from pyzbar.pyzbar import decode as pyzbar_decode
from ultralytics import YOLO

try:
    import zxingcpp
    ZXING_AVAILABLE = True
except ImportError:
    ZXING_AVAILABLE = False
    print("zxing not available, using pyzbar only")

print(f"Setup complete | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 4. Configuration


In [ ]:
# UPDATE THESE PATHS
MODEL_PATH = "/content/drive/MyDrive/Capstone Projects/ScanMatrix/weights/saved_model.pt"
IMAGE_PATH = "/content/drive/MyDrive/Capstone Projects/ScanMatrix/inference/IMG_6138.png"

# Detection settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIDENCE_THRESHOLD = 0.3
PADDING = 50
DETECTION_SCALES = [1.0, 2.0]
NMS_THRESHOLD = 0.5

print(f"Model: {MODEL_PATH}")
print(f"Image: {IMAGE_PATH}")

## 5. Detection

### About YOLO

YOLOv8 processes the entire image in a single forward pass, predicting bounding boxes and class probabilities simultaneously. This makes it fast enough for real time applications.

### Custom Training

The model was trained on a custom dataset to detect both 1D barcodes (UPC, EAN, Code128) and 2D barcodes (QR codes, Data Matrix).

### Multi Scale Detection

For distant barcodes, we run detection at multiple scales. This makes small barcodes easier to detect. Results are merged using Non Maximum Suppression to remove duplicates.

## 6. Decoding Pipeline

### Preprocessing Strategies

1. **Raw**: No preprocessing, for clear images
2. **Upscaling**: 2x, 3x, 4x bicubic interpolation for small barcodes
3. **CLAHE**: Local contrast enhancement for uneven lighting
4. **Sharpening**: Edge enhancement for blurry images
5. **Otsu**: Automatic binary threshold
6. **Sharpen + 5x**: Combined strategy for difficult cases

### Why Two Decoders?

Using both pyzbar and zxing increases overall success rate since they handle different edge cases.

### Rotation Handling

Both decoders try 0, 90, 180, and 270 degree rotations to handle angled barcodes.

In [ ]:
def get_image_quality(img):
    """Calculate sharpness using Laplacian variance. Higher = sharper."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def create_preprocessing_variants(crop):
    """Generate preprocessed versions of the crop for decoding attempts."""
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY) if len(crop.shape) == 3 else crop
    variants = [('original', crop)]
    
    # Upscaling
    for scale in [2, 3, 4]:
        upscaled = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        variants.append((f'upscale_{scale}x', upscaled))
    
    # CLAHE contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    variants.append(('clahe', cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)))
    
    # Sharpening
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])
    sharpened = cv2.filter2D(gray, -1, kernel)
    variants.append(('sharpen', cv2.cvtColor(sharpened, cv2.COLOR_GRAY2RGB)))
    
    # Otsu threshold
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(('binary', cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)))
    
    # Sharpen + upscale
    sharp_large = cv2.resize(cv2.cvtColor(sharpened, cv2.COLOR_GRAY2RGB),
                             None, fx=5, fy=5, interpolation=cv2.INTER_CUBIC)
    variants.append(('sharpen_5x', sharp_large))
    
    return variants


def try_decode(img, try_rotations=True):
    """Attempt to decode barcode using pyzbar and zxing with rotations."""
    angles = [0, 90, 180, 270] if try_rotations else [0]
    
    for angle in angles:
        if angle == 0:
            rotated = img
        else:
            h, w = img.shape[:2]
            center = (w // 2, h // 2)
            matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated = cv2.warpAffine(img, matrix, (w, h), borderMode=cv2.BORDER_REPLICATE)
        
        # Try pyzbar
        try:
            results = pyzbar_decode(rotated)
            if results:
                return {
                    'data': results[0].data.decode('utf-8'),
                    'format': results[0].type,
                    'decoder': 'pyzbar',
                    'rotation': angle
                }
        except:
            pass
        
        # Try zxing
        if ZXING_AVAILABLE:
            try:
                results = zxingcpp.read_barcodes(rotated)
                if results:
                    return {
                        'data': results[0].text,
                        'format': results[0].format.name,
                        'decoder': 'zxing',
                        'rotation': angle
                    }
            except:
                pass
    
    return None


print("Helper functions loaded")

## 7. Main Processing Function

### Processing Flow

1. Load and validate the input image
2. Run YOLO detection at multiple scales
3. Apply NMS to remove duplicate detections
4. For each detection, extract crop and try preprocessing variants until decoding succeeds
5. Annotate image with bounding boxes
6. Return decoded barcodes and annotated image

### Output

* **Green boxes**: Successfully decoded barcodes

In [ ]:
def detect_barcodes(model, img_rgb, confidence=0.3, scales=None):
    """Run YOLO detection at multiple scales and merge with NMS."""
    if scales is None:
        scales = [1.0]
    
    all_detections = []
    h_orig, w_orig = img_rgb.shape[:2]
    
    for scale in scales:
        if scale != 1.0:
            scaled = cv2.resize(img_rgb, (int(w_orig * scale), int(h_orig * scale)),
                               interpolation=cv2.INTER_CUBIC)
        else:
            scaled = img_rgb
        
        results = model.predict(source=scaled, conf=confidence, verbose=False, device=DEVICE)
        
        if results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = float(box.conf[0].item())
                x1, y1, x2, y2 = x1/scale, y1/scale, x2/scale, y2/scale
                all_detections.append({
                    'box': [int(x1), int(y1), int(x2), int(y2)],
                    'confidence': conf,
                    'scale': scale
                })
    
    # Apply NMS
    if len(all_detections) > 1:
        from torchvision.ops import nms
        boxes = torch.tensor([d['box'] for d in all_detections], dtype=torch.float32)
        scores = torch.tensor([d['confidence'] for d in all_detections], dtype=torch.float32)
        keep = nms(boxes, scores, NMS_THRESHOLD)
        all_detections = [all_detections[i] for i in keep.tolist()]
    
    return all_detections


def process_image(model, image_path, confidence=0.1, scales=None, padding=50, show_details=True):
    """Main pipeline: detect barcodes and decode each detection."""
    if scales is None:
        scales = [1.0]
    
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return {'status': 'ERROR', 'message': f'Could not load: {image_path}'}
    
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    if show_details:
        print(f"Image: {os.path.basename(image_path)} ({w}x{h}px)")
    
    detections = detect_barcodes(model, img_rgb, confidence, scales)
    
    if not detections:
        if show_details:
            print("No barcodes detected")
        return {'status': 'NO_DETECTION', 'barcodes': [], 'image': img_rgb, 'detection_count': 0}
    
    if show_details:
        print(f"Found {len(detections)} detection(s)")
    
    decoded_list = []
    output_img = img_rgb.copy()
    
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det['box']
        conf = det['confidence']
        
        # Extract crop with padding
        px1, py1 = max(0, x1 - padding), max(0, y1 - padding)
        px2, py2 = min(w, x2 + padding), min(h, y2 + padding)
        crop = img_rgb[py1:py2, px1:px2]
        
        if crop.size == 0:
            continue
        
        sharpness = get_image_quality(crop)
        
        if show_details:
            print(f"  Detection {i+1}: {crop.shape[1]}x{crop.shape[0]}px, conf={conf:.2f}, sharpness={sharpness:.0f}")
        
        variants = create_preprocessing_variants(crop)
        decoded = False
        
        for variant_name, variant_img in variants:
            result = try_decode(variant_img)
            if result:
                if show_details:
                    print(f"      Decoded ({result['decoder']}+{variant_name}): {result['data']}")
                
                decoded_list.append({
                    'data': result['data'],
                    'format': result['format'],
                    'method': f"{result['decoder']}+{variant_name}",
                    'confidence': conf,
                    'box': det['box']
                })
                
                cv2.rectangle(output_img, (x1, y1), (x2, y2), (0, 255, 0), 4)
                label = result['data'][:25] + "..." if len(result['data']) > 25 else result['data']
                cv2.putText(output_img, label, (x1, max(y1-10, 25)),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                decoded = True
                break
        
        if not decoded:
            if show_details:
                print(f"      Failed to decode")
            cv2.rectangle(output_img, (x1, y1), (x2, y2), (255, 0, 0), 3)
    
    status = 'SUCCESS' if decoded_list else 'DECODE_FAILED'
    return {
        'status': status,
        'barcodes': decoded_list,
        'image': output_img,
        'detection_count': len(detections)
    }


print("Pipeline ready")

## 8. Execution

In [ ]:
# Load model
model = YOLO(MODEL_PATH)
model.to(DEVICE)
print(f"Model loaded")

In [ ]:
# Process image
result = process_image(
    model=model,
    image_path=IMAGE_PATH,
    confidence=CONFIDENCE_THRESHOLD,
    scales=DETECTION_SCALES,
    padding=PADDING,
    show_details=True
)

print("\n" + "="*60)
print(f"RESULT: {result['status']}")
print("="*60)

if result['status'] == 'SUCCESS':
    print(f"\nDecoded {len(result['barcodes'])} barcode(s):")
    for i, bc in enumerate(result['barcodes'], 1):
        print(f"  {i}. {bc['data']}")
        print(f"     Format: {bc['format']} | Method: {bc['method']}")
elif result['status'] == 'DECODE_FAILED':
    print(f"\nDetected {result['detection_count']} barcode(s) but could not decode")

plt.figure(figsize=(14, 10))
plt.imshow(result['image'])
plt.title(f"Result: {result['status']} | Decoded: {len(result.get('barcodes', []))}")
plt.axis('off')
plt.tight_layout()
plt.show()

## 9. Batch Processing

In [ ]:
def batch_process(model, image_dir, confidence=0.1, scales=None, max_images=None):
    """Process all images in a directory and return statistics."""
    if scales is None:
        scales = [1.0]
    
    extensions = ('.png', '.jpg', '.jpeg', '.bmp')
    images = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(extensions)])
    
    if max_images:
        images = images[:max_images]
    
    print(f"Processing {len(images)} images...\n")
    
    results = []
    start = time()
    
    for img_name in images:
        img_path = os.path.join(image_dir, img_name)
        result = process_image(model, img_path, confidence, scales, show_details=False)
        
        status_tag = "OK" if result['status'] == 'SUCCESS' else "FAIL"
        decoded_count = len(result.get('barcodes', []))
        det_count = result.get('detection_count', 0)
        
        print(f"[{status_tag}] {img_name}: {decoded_count}/{det_count}")
        
        results.append({
            'image': img_name,
            'status': result['status'],
            'decoded': decoded_count,
            'detected': det_count
        })
    
    elapsed = time() - start
    success_count = sum(1 for r in results if r['status'] == 'SUCCESS')
    total_decoded = sum(r['decoded'] for r in results)
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Images: {len(results)}")
    print(f"Success rate: {success_count}/{len(results)} ({100*success_count/len(results):.0f}%)")
    print(f"Total decoded: {total_decoded}")
    print(f"Time: {elapsed:.1f}s")
    
    return results



## 10. Results and Evaluation

### Performance

| Condition | Success Rate |
|-----------|-------------|
| Close, good lighting | 80-95% |
| Medium distance | 60-80% |
| Far, low light | 30-50% |

### Key Factors

1. **Sharpness**: Values above 100 typically decode successfully
2. **Barcode Size**: Width > 200px decodes more reliably
3. **Lighting**: Even, diffuse lighting works best
4. **Angle**: Perpendicular to barcode is optimal


### Supported Formats

**1D:** Code 128, Code 39, EAN-13, EAN-8, UPC-A, UPC-E, ITF, Codabar

**2D:** QR Code, Data Matrix, PDF417, Aztec

---

## References

1. Ultralytics YOLOv8: https://docs.ultralytics.com/
2. pyzbar: https://github.com/NaturalHistoryMuseum/pyzbar
3. zxing-cpp: https://github.com/zxing-cpp/zxing-cpp
4. OpenCV: https://docs.opencv.org/